# 城市财政缺口分析

本notebook进行城市财政数据的探索性数据分析，包括：
1. 计算预算缺口和相对GDP缺口
2. 计算收入和支出增长率
3. 特定年份缺口最大/最小城市列表

In [2]:
# 导入库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams['font.sans-serif'] = ['SimHei']  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False

In [3]:
# 读取清洗后的数据
data_path = './data_clean/cleaned_panel_data_v2.csv'
df = pd.read_csv(data_path)
print("数据读取完成")
print(f"数据形状: {df.shape}")
df.head()

数据读取完成
数据形状: (684, 6)


,城市,年度,财政收入(亿元),财政支出(亿元),住户存款余额(亿元),地区生产总值(亿元)
0,上海,2006,1576.07,1795.57,8730.00,10825.4
1,上海,2007,2074.48,2181.68,8745.22,13179.8
2,上海,2008,2358.75,2593.92,11464.15,14877.1
3,上海,2009,2540.30,2989.65,13707.32,16181.4
4,上海,2010,2873.58,3302.89,15650.24,18319.6


In [4]:
# 任务1: 计算预算缺口
df['gap'] = df['财政支出(亿元)'] - df['财政收入(亿元)']
df['gap_to_gdp'] = df['gap'] / df['地区生产总值(亿元)']
print("预算缺口计算完成")
df[['城市', '年度', 'gap', 'gap_to_gdp']].head()

预算缺口计算完成


,城市,年度,gap,gap_to_gdp
0,上海,2006,219.50,0.020276
1,上海,2007,107.20,0.008134
2,上海,2008,235.17,0.015808
3,上海,2009,449.35,0.027770
4,上海,2010,429.31,0.023434


In [9]:
# 保存gap分析结果
gap_output_path = './output/gap_analysis.csv'
gap_df = df[['城市', '年度', 'gap', 'gap_to_gdp']].copy()
gap_df.to_csv(gap_output_path, index=False, encoding='utf-8-sig')
print(f"gap分析结果已保存到: {gap_output_path}")

gap分析结果已保存到: ./output/gap_analysis.csv


In [5]:
# 任务2: 计算收入和支出增长率
df_sorted = df.sort_values(['城市', '年度'])
df_sorted['income_growth'] = df_sorted.groupby('城市')['财政收入(亿元)'].pct_change()
df_sorted['expend_growth'] = df_sorted.groupby('城市')['财政支出(亿元)'].pct_change()
print("增长率计算完成")
df_sorted[['城市', '年度', '财政收入(亿元)', 'income_growth', '财政支出(亿元)', 'expend_growth']].head(10)

增长率计算完成


,城市,年度,财政收入(亿元),income_growth,财政支出(亿元),expend_growth
0,上海,2006,1576.07,NaN,1795.57,NaN
1,上海,2007,2074.48,0.316236,2181.68,0.215035
2,上海,2008,2358.75,0.137032,2593.92,0.188955
3,上海,2009,2540.30,0.076969,2989.65,0.152561
4,上海,2010,2873.58,0.131197,3302.89,0.104775
5,上海,2011,3429.83,0.193574,3914.88,0.185289
6,上海,2012,3743.71,0.091515,4184.02,0.068748
7,上海,2013,4109.51,0.097711,4528.61,0.082359
8,上海,2014,4585.55,0.115839,4923.44,0.087186
9,上海,2015,5519.50,0.203672,6191.56,0.257568


In [10]:
# 保存增长率结果
growth_output_path = './output/growth_rates.csv'
growth_df = df_sorted[['城市', '年度', 'income_growth', 'expend_growth']].copy()
growth_df.to_csv(growth_output_path, index=False, encoding='utf-8-sig')
print(f"增长率结果已保存到: {growth_output_path}")

增长率结果已保存到: ./output/growth_rates.csv


In [11]:
# 任务3: 特定年份gap_to_gdp最大和最小的城市
# 读取gap分析结果
gap_df = pd.read_csv('./output/gap_analysis.csv')
years = [2006, 2010, 2014, 2018, 2022]
for year in years:
    year_data = gap_df[gap_df['年度'] == year]
    if not year_data.empty:
        max_city = year_data.loc[year_data['gap_to_gdp'].idxmax(), '城市']
        min_city = year_data.loc[year_data['gap_to_gdp'].idxmin(), '城市']
        max_value = year_data['gap_to_gdp'].max()
        min_value = year_data['gap_to_gdp'].min()
        print(f"{year}年: gap_to_gdp最大城市 - {max_city} ({max_value:.4f}), 最小城市 - {min_city} ({min_value:.4f})")
    else:
        print(f"{year}年: 无数据")

2006年: gap_to_gdp最大城市 - 西宁 (0.0716), 最小城市 - 乌鲁木齐 (-0.0115)
2010年: gap_to_gdp最大城市 - 西宁 (0.1184), 最小城市 - 杭州 (-0.0092)
2014年: gap_to_gdp最大城市 - 拉萨 (0.3013), 最小城市 - 杭州 (-0.0072)
2018年: gap_to_gdp最大城市 - 拉萨 (0.3514), 最小城市 - 杭州 (-0.0080)
2022年: gap_to_gdp最大城市 - 拉萨 (0.3784), 最小城市 - 杭州 (0.0049)
